<a href="https://colab.research.google.com/github/gopalpatil15/data-engineering-practice/blob/main/04_payment_etl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import sqlite3
from sqlalchemy import create_engine

In [3]:
df_txn = pd.read_csv('/content/transactions_20260624.csv')

In [4]:
print(df_txn.shape)
print(df_txn.dtypes)
print(df_txn.isna().sum())

(10000, 11)
txn_id             object
timestamp          object
sender_bank        object
sender_account     object
receiver_bank      object
upi_app            object
amount            float64
status             object
merchant           object
city               object
category           object
dtype: object
txn_id              0
timestamp           0
sender_bank         0
sender_account      0
receiver_bank       0
upi_app             0
amount            501
status              0
merchant            0
city                0
category            0
dtype: int64


In [5]:
df_txn.head(5)

,txn_id,timestamp,sender_bank,sender_account,receiver_bank,upi_app,amount,status,merchant,city,category
0,UPI000001,2026-01-05 15:57:00,PNB,ACC81482,SBI,AmazonPay,1260.29,REVERSED,Edwin-Chaudhari,Agra,Food
1,UPI000002,2026-01-10 01:11:00,PNB,ACC95181,PNB,AmazonPay,-10939.71,REVERSED,"Sengupta, Bath and Buch",Bulandshahr,Transport
2,UPI000003,2026-01-08 06:23:00,PNB,ACC65392,ICICI,Paytm,13916.75,FAILED,Pall Inc,Avadi,Transport
3,UPI000004,2026-01-16 15:41:00,Kotak,ACC44671,BOB,PhonePe,16836.36,REVERSED,Vig LLC,Pallavaram,Transfer
4,UPI000005,2026-01-29 14:38:00,Kotak,ACC57400,Kotak,GPay,46115.55,SUCCESS,Kale-Mandal,Rajpur Sonarpur,Food


In [6]:
df_txn['amount'].dtype

dtype('float64')

# Cell 2 → Clean:
         - pd.to_numeric on amount
         - drop nulls on txn_id and amount
         - drop negatives
         - drop duplicates on txn_id
         - strip + upper on status
         - reset_index

In [7]:
df_txn['amount'] = pd.to_numeric(df_txn['amount'], errors='coerce')
df_txn = df_txn.dropna(subset = ['txn_id','amount'] )
df_txn = df_txn[df_txn['amount'] > 0]
df_txn['status'] = df_txn['status'].str.strip().str.upper()
df_txn = df_txn.reset_index(drop=True)
df_txn = df_txn.drop_duplicates(subset=['txn_id'], keep='first')

In [8]:
print(f"Before: 10,000 | After: {len(df_txn)} | Dropped: {10000 - len(df_txn)}")

Before: 10,000 | After: 9046 | Dropped: 954


# Cell 3 — Feature Engineering


- parse timestamp to datetime
- extract hour, day_name, month, quarter
- create is_weekend (True if Saturday or Sunday)
- print sample of 3 rows showing new columns

In [9]:
df_txn = df_txn.copy()

df_txn.columns = (df_txn.columns
                    .str.lower()
                    .str.replace(' ', '_')
                    .str.replace('-', '_'))

print("Fixed columns:", df_txn.columns.tolist())

df_txn['timestamp'] = pd.to_datetime(df_txn['timestamp'])

df_txn['hour'] = df_txn['timestamp'].dt.hour
df_txn['day_name'] = df_txn['timestamp'].dt.day_name()
df_txn['month'] = df_txn['timestamp'].dt.month
df_txn['quarter'] = df_txn['timestamp'].dt.quarter
df_txn['is_weekend'] = df_txn['timestamp'].dt.day_name().isin(['Saturday', 'Sunday'])

print(f"\nSample:\n{df_txn[['timestamp','hour','day_name','month','quarter','is_weekend']].head(3)}")

Fixed columns: ['txn_id', 'timestamp', 'sender_bank', 'sender_account', 'receiver_bank', 'upi_app', 'amount', 'status', 'merchant', 'city', 'category']

Sample:
            timestamp  hour  day_name  month  quarter  is_weekend
0 2026-01-05 15:57:00    15    Monday      1        1       False
1 2026-01-08 06:23:00     6  Thursday      1        1       False
2 2026-01-16 15:41:00    15    Friday      1        1       False


# Cell 4 → numpy
         - amount_tier (HIGH/MEDIUM/LOW)
         - is_outlier (99th percentile)
         - flag_suspicious (amount > 40000 AND status != SUCCESS)


In [15]:
df_txn['amount_tier'] = np.where(df_txn['amount'] > 10000, 'HIGH',
                                 np.where(df_txn['amount'] > 1000, 'MEDIUM', 'LOW'))
amount_p99_threshold = np.percentile(df_txn['amount'], 99)
df_txn['is_outlier'] = np.where(df_txn['amount'] > amount_p99_threshold, True, False)

df_txn['flag_suspicious'] = np.where((df_txn['amount'] > 40000) & (df_txn['status'] != 'SUCCESS'), True, False)


print("Amount Tier Distribution:")
print(df_txn['amount_tier'].value_counts())

print("\nOutlier Count:")
print(df_txn['is_outlier'].value_counts())

print("\nSuspicious Transactions Count:")
print(df_txn['flag_suspicious'].value_counts())


Amount Tier Distribution:
amount_tier
HIGH      7236
MEDIUM    1626
LOW        184
Name: count, dtype: int64

Outlier Count:
is_outlier
False    8955
True       91
Name: count, dtype: int64

Suspicious Transactions Count:
flag_suspicious
False    7698
True     1348
Name: count, dtype: int64


# Cell 5 → groupby KPIs
1. Total amount by city — top 5
2. Success rate by upi_app
3. Average amount by category
4. Transaction count by hour — peak hours
5. Status distribution


In [28]:
city = df_txn.groupby('city')['amount'].sum().sort_values(ascending=False).head(5)

success_rate = df_txn[df_txn['status'] == 'SUCCESS'].groupby('upi_app')['txn_id'].count() / df_txn.groupby('upi_app')['txn_id'].count() * 100

avg_amount_by_category = df_txn.groupby('category')['amount'].mean()

transaction_count_by_hour = df_txn.groupby('hour')['txn_id'].count()

status_distribution = df_txn['status'].value_counts()


print("=" * 45)
print("        TRANSACTION KPI DASHBOARD       ")
print("=" * 45)

print("\n TOP 5 CITIES BY TOTAL REVENUE")
print("-" * 45)
for c, rev in city.items():
    print(f"  • {c:<18} : ₹{rev:,.2f}")

print("\n SUCCESS RATE BY UPI APP")
print("-" * 45)
for app, rate in success_rate.items():
    print(f"  • {app:<18} : {rate:.2f}%")

print("\n  AVERAGE TRANSACTION AMOUNT BY CATEGORY")
print("-" * 45)
for cat, avg in avg_amount_by_category.items():
    print(f"  • {cat:<18} : ₹{avg:,.2f}")

print("\n TRANSACTION STATUS DISTRIBUTION")
print("-" * 45)
for stat, count in status_distribution.items():
    print(f"  • {stat:<18} : {count:,} txns")

print("\n PEAK TRANSACTION HOURS (TOP 5)")
print("-" * 45)
# Sorting the hour counts to show the busiest times first
for hr, count in transaction_count_by_hour.sort_values(ascending=False).head(5).items():
    print(f"  • Hour {hr:02d}:00          : {count:,} txns")

print("=" * 45)

        TRANSACTION KPI DASHBOARD       

 TOP 5 CITIES BY TOTAL REVENUE
---------------------------------------------
  • Ghaziabad          : ₹1,610,235.73
  • Bhiwandi           : ₹1,297,470.04
  • Khandwa            : ₹1,224,721.98
  • Aurangabad         : ₹1,220,865.95
  • Allahabad          : ₹1,192,055.16

 SUCCESS RATE BY UPI APP
---------------------------------------------
  • AmazonPay          : 24.90%
  • BHIM               : 24.82%
  • GPay               : 23.77%
  • Paytm              : 25.86%
  • PhonePe            : 25.43%

  AVERAGE TRANSACTION AMOUNT BY CATEGORY
---------------------------------------------
  • Bills              : ₹24,872.98
  • Food               : ₹24,435.31
  • Shopping           : ₹24,482.77
  • Transfer           : ₹25,811.50
  • Transport          : ₹24,887.66

 TRANSACTION STATUS DISTRIBUTION
---------------------------------------------
  • REVERSED           : 2,311 txns
  • FAILED             : 2,260 txns
  • SUCCESS            : 2,257 txn